In [1]:
import pandas as pd
import numpy as np
import os

# ==========================================================
# config
# ==========================================================

FILE_PATH = "drugcomb_288_146.csv"
# FILE_PATH = "drugcombDB_126_67.csv"
RANDOM_STATE = 42

# 1.0 means no sampling
SAMPLE_FRAC = 0.3 # for drugcomb
# SAMPLE_FRAC = 1.0 # for drugcombDB

# --------------------------
# target_drugs selection
# --------------------------
TARGET_DRUG_RATIO = 0.1           # target_drugs = total_drugs * TARGET_DRUG_RATIO
TARGET_DRUG_SELECTION = "uniform" # 'high' | 'low' | 'uniform'
# high: pick drugs with high frequency
# low: pick drugs with low frequency
# uniform: pick drugs uniformly based on frequency ranking

# Rule: If any drug in a sample belongs to the "target_drugs" list, 
# then the sample belongs to the target domain; otherwise, it belongs to the source domain.

# --------------------------
# Source Domain Internal Split (Stratified by Cell Line)
# --------------------------
# Output: train/val/test (all labeled)
SOURCE_VAL_RATIO_PER_CELL = 0.2
SOURCE_TEST_RATIO_PER_CELL = 0.2
# train_ratio = 1 - val_ratio - test_ratio

# --------------------------
# Target Domain Internal Split (Stratified by Cell Line)
# --------------------------
# Output: unlabeled_train / labeled_test
TARGET_UNLABELED_RATIO_PER_CELL = 0.9

# --------------------------
# Threshold for Cell Line Samples
# --------------------------
# If a cell line has too few samples, skipping the split (put all into the main training set)
MIN_SAMPLES_THRESHOLD = 3  # Suggest >=3 for 3-way split; >=2 for 2-way split


# ==========================================================
# Utility Functions
# ==========================================================

def get_drug_counts(df: pd.DataFrame, drugA_col: str, drugB_col: str) -> pd.DataFrame:
    """
    Count total occurrences of each drug in drugA and drugB columns, 
    and sort them in descending order.
    """
    cA = df[drugA_col].value_counts()
    cB = df[drugB_col].value_counts()
    counts = cA.add(cB, fill_value=0).reset_index()
    counts.columns = ["drug", "sample_count"]
    return counts.sort_values("sample_count", ascending=False).reset_index(drop=True)


def select_target_drugs(drug_counts: pd.DataFrame, target_ratio: float, mode: str) -> tuple[set, str]:
    """
    Select target domain drug set based on the specified strategy.
    """
    n_total = len(drug_counts)
    n_target = max(1, int(n_total * target_ratio))

    if mode == "high":
        target = drug_counts.iloc[:n_target]["drug"].tolist()
        note = f"Strategy=high: Selected top {n_target}/{n_total} high-frequency drugs as target drugs"

    elif mode == "low":
        target = drug_counts.iloc[-n_target:]["drug"].tolist()
        note = f"Strategy=low: Selected bottom {n_target}/{n_total} low-frequency drugs as target drugs"

    elif mode == "uniform":
        # Uniform sampling based on ranking
        idx = np.linspace(0, n_total - 1, n_target, dtype=int)
        idx = np.unique(idx)
        target = drug_counts.iloc[idx]["drug"].tolist()
        note = f"Strategy=uniform: Sampled {len(target)} drugs uniformly from {n_total} drugs (sorted by frequency)"

    else:
        raise ValueError("TARGET_DRUG_SELECTION must be 'high', 'low', or 'uniform'")

    return set(target), note


def split_source_by_cell_3way(source_df: pd.DataFrame,
                             cell_col: str,
                             val_ratio: float,
                             test_ratio: float,
                             min_threshold: int,
                             seed: int):
    """
    Source Domain: Split into train/val/test (fully labeled) within each cell line.
    If samples < min_threshold, move all samples to the training set.
    """
    assert val_ratio + test_ratio < 1.0, "Source val_ratio + test_ratio must be < 1"

    train_list, val_list, test_list = [], [], []

    for cell, g in source_df.groupby(cell_col):
        g = g.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n = len(g)

        if n < min_threshold:
            train_list.append(g)
            continue

        n_val = int(round(n * val_ratio))
        n_test = int(round(n * test_ratio))

        # Ensure at least 1 sample for val/test if n >= 3
        if n >= 3:
            n_val = max(1, n_val)
            n_test = max(1, n_test)

        # Ensure train set is not empty
        if n_val + n_test >= n:
            if n >= 3:
                n_val, n_test = 1, 1
            else:
                n_val, n_test = 1, 0

        n_train = n - n_val - n_test
        if n_train <= 0:
            n_train = 1
            if n_val > 0:
                n_val -= 1
            elif n_test > 0:
                n_test -= 1

        train_list.append(g.iloc[:n_train].copy())
        if n_val > 0:
            val_list.append(g.iloc[n_train:n_train + n_val].copy())
        if n_test > 0:
            test_list.append(g.iloc[n_train + n_val:n_train + n_val + n_test].copy())

    train_df = pd.concat(train_list, ignore_index=True) if train_list else source_df.iloc[0:0].copy()
    val_df = pd.concat(val_list, ignore_index=True) if val_list else source_df.iloc[0:0].copy()
    test_df = pd.concat(test_list, ignore_index=True) if test_list else source_df.iloc[0:0].copy()
    return train_df, val_df, test_df


def split_target_by_cell_2way(target_df: pd.DataFrame,
                             cell_col: str,
                             unlabeled_ratio: float,
                             min_threshold: int,
                             seed: int):
    """
    Target Domain: Split into unlabeled_train / labeled_test within each cell line.
    If samples < min_threshold, move all samples to unlabeled_train.
    """
    unlabeled_list, test_list = [], []

    for cell, g in target_df.groupby(cell_col):
        g = g.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n = len(g)

        if n < min_threshold:
            unlabeled_list.append(g)
            continue

        n_unlabeled = int(round(n * unlabeled_ratio))
        n_unlabeled = max(1, n_unlabeled)
        if n_unlabeled >= n:
            n_unlabeled = n - 1  # Ensure test set is not empty (for n >= 2)

        unlabeled_list.append(g.iloc[:n_unlabeled].copy())
        test_list.append(g.iloc[n_unlabeled:].copy())

    unlabeled_df = pd.concat(unlabeled_list, ignore_index=True) if unlabeled_list else target_df.iloc[0:0].copy()
    test_df = pd.concat(test_list, ignore_index=True) if test_list else target_df.iloc[0:0].copy()
    return unlabeled_df, test_df


def brief_stats(name: str, df: pd.DataFrame, cell_col: str):
    """Print brief summary statistics of the subset."""
    print(f"{name}: Samples={len(df):,} | Cell Lines={df[cell_col].nunique():,}")


# ==========================================================
# Main Execution Flow
# ==========================================================

# 1) Read data
data = pd.read_csv(FILE_PATH, low_memory=False)
print(f"[1] Data loading complete: {FILE_PATH}, Total Samples={len(data):,}")

# 2) Subsampling (Optional)
if SAMPLE_FRAC < 1.0:
    data = data.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"[2] Sampling: Ratio={SAMPLE_FRAC:.2f}, Samples after sampling={len(data):,}")
else:
    print(f"[2] Sampling: SAMPLE_FRAC=1.0, using full dataset")

# 3) Define column names
cell_line_col = "cell_line_id"
drugA_col = "drug_row"
drugB_col = "drug_col"

for col in [cell_line_col, drugA_col, drugB_col]:
    if col not in data.columns:
        raise ValueError(f"Missing required column: {col}. Available columns: {list(data.columns)}")

# 4) Count drug frequency and select target domain drugs
drug_counts = get_drug_counts(data, drugA_col, drugB_col)
target_drugs, selection_note = select_target_drugs(drug_counts, TARGET_DRUG_RATIO, TARGET_DRUG_SELECTION)

print("\n[3] Drug Frequency Statistics:")
print(f"    Total number of drugs={len(drug_counts):,}")
print("    Top 10 High Frequency:")
print(drug_counts.head(10))
print("    Bottom 10 Low Frequency:")
print(drug_counts.tail(10))

print("\n[4] Target Drug Selection:")
print("   ", selection_note)
print(f"    Target drugs count={len(target_drugs):,}, Example={list(target_drugs)[:10]}")

# 5) Split Source/Target domain based on drug presence
is_target = data[drugA_col].isin(target_drugs) | data[drugB_col].isin(target_drugs)
target_data = data[is_target].reset_index(drop=True)
source_data = data[~is_target].reset_index(drop=True)

print("\n[5] Drug-based Domain Split Complete:")
print(f"    Target domain samples={len(target_data):,} | Cell lines={target_data[cell_line_col].nunique():,}")
print(f"    Source domain samples={len(source_data):,} | Cell lines={source_data[cell_line_col].nunique():,}")

# Safety check: No target drugs should exist in the source domain
leak = source_data[drugA_col].isin(target_drugs) | source_data[drugB_col].isin(target_drugs)
assert not leak.any(), "Target drugs detected in source domain! Check split logic."

# 6) Source Domain Internal Split: train/val/test
print("\n[6] Source domain internal split (stratified by cell line): train/val/test")
source_train, source_val, source_test = split_source_by_cell_3way(
    source_data,
    cell_col=cell_line_col,
    val_ratio=SOURCE_VAL_RATIO_PER_CELL,
    test_ratio=SOURCE_TEST_RATIO_PER_CELL,
    min_threshold=MIN_SAMPLES_THRESHOLD,
    seed=RANDOM_STATE
)

# 7) Target Domain Internal Split: unlabeled_train / labeled_test
print("\n[7] Target domain internal split (stratified by cell line): unlabeled_train / labeled_test")
target_train_unlabeled, target_test_labeled = split_target_by_cell_2way(
    target_data,
    cell_col=cell_line_col,
    unlabeled_ratio=TARGET_UNLABELED_RATIO_PER_CELL,
    min_threshold=max(2, MIN_SAMPLES_THRESHOLD - 1), 
    seed=RANDOM_STATE
)

# 8) Consistency Check
assert len(source_train) + len(source_val) + len(source_test) == len(source_data), "Source domain count mismatch!"
assert len(target_train_unlabeled) + len(target_test_labeled) == len(target_data), "Target domain count mismatch!"

print("\n[8] Consistency Check Passed:")
print("    - Target drugs only exist in the target domain samples.")
print("    - Source domain train/val/test samples are conserved.")
print("    - Target domain unlabeled/test samples are conserved.")

# 9) Save CSV files
os.makedirs("dataset", exist_ok=True)
source_train.to_csv("dataset/source_train_labeled.csv", index=False)
source_val.to_csv("dataset/source_val_labeled.csv", index=False)
source_test.to_csv("dataset/source_test_labeled.csv", index=False)
target_train_unlabeled.to_csv("dataset/target_train_unlabeled.csv", index=False)
target_test_labeled.to_csv("dataset/target_test_labeled.csv", index=False)

print("\n[9] Files saved successfully:")
brief_stats("Source Train (Labeled)", source_train, cell_line_col)
brief_stats("Source Val (Labeled)", source_val, cell_line_col)
brief_stats("Source Test (Labeled)", source_test, cell_line_col)
brief_stats("Target Train (Unlabeled)", target_train_unlabeled, cell_line_col)
brief_stats("Target Test (Labeled)", target_test_labeled, cell_line_col)
print("\nTarget drug selection strategy:", selection_note)

[1] Data loading complete: drugcomb_288_146.csv, Total Samples=382,022
[2] Sampling: Ratio=0.30, Samples after sampling=114,607

[3] Drug Frequency Statistics:
    Total number of drugs=288
    Top 10 High Frequency:
                drug  sample_count
0          Dasatinib        3113.0
1          Sorafenib        3103.0
2          Sunitinib        3048.0
3         Vorinostat        3046.0
4          Lapatinib        3033.0
5       temozolomide        2683.0
6         Bortezomib        2623.0
7  ADM hydrochloride        2238.0
8         Crizotinib        2124.0
9     5-Fluorouracil        2115.0
    Bottom 10 Low Frequency:
                   drug  sample_count
278  4-Hydroxytamoxifen          25.0
279           NSC169534          24.0
280           TNF-Alpha          24.0
281        Temsirolimus          23.0
282             AZD2461          22.0
283          20439-47-8          22.0
284           NSC707389          20.0
285      SCHEMBL3315478          20.0
286     Folfox protocol    